<a href="https://colab.research.google.com/github/rahavi-r31/ExporterAI_Chapter_68_analytics/blob/colab/final_cleaning_script.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Importing files and checking the formats and NULL vaules

In [ ]:
pip install rapidfuzz

In [ ]:
import pandas as pd
import numpy as np
import re
from rapidfuzz import process, fuzz

In [ ]:
data1=pd.read_csv()

instruction to add file address

In [ ]:
data1.info()

In [ ]:
# Selection of specific columns
data = data1[['SB DATE', 'FOB INR', 'QUANTITY', 'IMPORTER', 'EXPORTER', 'HS CODE', 'PRODUCT DESCRIPTION', 'FOREIGN COUNTRY', 'FOREIGN PORT', 'INDIAN PORT', 'IEC', 'CHAPTER']]

In [ ]:
#  Data["column name"] = data["column name"].astype(int64 or float64)
data["SB DATE"] = pd.to_datetime(data["SB DATE"], errors="coerce", dayfirst=True)
# it is expected that date values are changed into one format either full numeric or alphanumeric in raw file
data["FOB INR"] = (
    data["FOB INR"]
    .astype(str)
    .str.strip()
    .str.replace(",", "", regex=False) #changes , to empty space like 43,452 -> 43452
    .replace("UNKNOWN", None)
    .astype(float)
)

In [ ]:
data.describe()

In [ ]:
data.isnull().sum()

In [ ]:
data[["QUANTITY", "FOB INR"]] = data[["QUANTITY", "FOB INR"]].fillna(0)

In [ ]:
data[data["IEC"].isnull()]
mask = data["IEC"].isna() & data1["Item_Category_Description"].notna() #checks if iec value is present in the particular column and makes a list of true or false
data.loc[mask, "IEC"] = data1.loc[mask, "Item_Category_Description"] #copy the value to iec

In [ ]:
# List of values that are not acceptable
invalid_values = ["#REF!", "#NAME?", "nana", "NA", "#N/A", "Na"]

# Replace in the whole DataFrame
data = data.replace(invalid_values, "UNKNOWN")
data = data.fillna("UNKNOWN")

In [ ]:
data.isnull().sum()

In [ ]:
unknown_counts = (data == "UNKNOWN").sum()
print(unknown_counts)

In [ ]:
data.describe()

In [ ]:
data.info()

# Column wise cleaning

## foreign country

In [ ]:
# --- Reference file ---
reference_file = r"/content/countries.csv"
ref_df = pd.read_csv(
    reference_file,
    sep=";",
    quotechar='"',
    encoding="latin1"
)

# Normalise reference names
ref_df["name_normalized"] = (
    ref_df["name"]
    .str.strip()
    .str.lower()
    .str.replace(r"[.,'’-]", "", regex=True)   # remove dots, commas, apostrophes, hyphens
    .str.replace(r"\s+", " ", regex=True)      # normalise multiple spaces
)

# --- Manual overrides ---
manual_map = {
    "united states": "United States of America",
    "u.s.a": "United States of America",
    "usa": "United States of America",
    "us": "United States of America",
    "america": "United States of America",
    "turkey": "Türkiye",
    "kyr": "KYRGYZ REPUBLIC",
    "uae": "United Arab Emirates"
}

# --- Cleaning function ---
def enforce_iso_names(data, country_col, ref_df, iso_col="iso3", ref_name_col="name", score_threshold=85):
    choices = ref_df[ref_name_col].dropna().unique().tolist()
    choices_lower = [c.lower() for c in choices]

    match_stats = {"manual": 0, "exact": 0, "fuzzy": 0, "unmatched": 0}

    def normalise_text(x: str) -> str:
        """Remove punctuation, multiple spaces, lowercase etc."""
        x = x.strip().lower()
        x = re.sub(r"[.,'’-]", "", x)     # remove punctuation
        x = re.sub(r"\s+", " ", x)        # collapse multiple spaces
        return x

    def best_match(x):
        if pd.isna(x):
            match_stats["unmatched"] += 1
            return "Unmatched"

        x_norm = normalise_text(str(x))

        # Step 1: Manual overrides
        if x_norm in manual_map:
            match_stats["manual"] += 1
            return manual_map[x_norm]

        # Step 2: Exact match
        if x_norm in ref_df["name_normalized"].values:
            match_stats["exact"] += 1
            return ref_df.loc[ref_df["name_normalized"] == x_norm, ref_name_col].values[0]

        # Step 3: Fuzzy match
        match, score, _ = process.extractOne(x_norm, choices_lower)
        if score >= score_threshold:
            idx = choices_lower.index(match)
            match_stats["fuzzy"] += 1
            return choices[idx]

        # Step 4: Unmatched
        match_stats["unmatched"] += 1
        return "Unmatched"

    # Apply
    data["destination_country_cleaned"] = data[country_col].apply(best_match)

    # Merge ISO codes
    data = data.merge(
        ref_df[[iso_col, ref_name_col]],
        left_on="destination_country_cleaned",
        right_on=ref_name_col,
        how="left"
    )

    data = data.rename(columns={iso_col: "iso_code"})

    print("\n✅ Cleaning complete! Every country name standardised to ISO reference.")
    print("\n📊 Summary of match methods:")
    for k, v in match_stats.items():
        print(f" - {k.title()} matches: {v}")

    return data


# --- Run cleaning ---
cleaned_df = enforce_iso_names(data, "FOREIGN COUNTRY", ref_df)

# --- Show unmatched ---
unmatched = cleaned_df[cleaned_df["destination_country_cleaned"] == "Unmatched"]
if not unmatched.empty:
    print("\n⚠️ Some rows could not be matched to ISO names. Please review:")
    print(unmatched[["FOREIGN COUNTRY"]].drop_duplicates())


## indian port

In [ ]:
reference_file = r"/content/ports.xlsx"
ref_df = pd.read_excel(reference_file)   # reference with only "Name"

# Normalise reference names for matching
ref_df["name_normalized"] = ref_df["Name"].str.strip().str.lower()

def clean_expression(text: str) -> str:
    """
    Removes unwanted expressions like:
    - text after ';' or '-'
    - brackets ()
    - multiple spaces
    """
    if pd.isna(text):
        return ""

    text = str(text).strip().lower()

    # Remove anything after ';' or '-'
    text = re.split(r"[;\-]", text)[0]

    # Remove things inside brackets
    text = re.sub(r"\(.*?\)", "", text)

    # Collapse multiple spaces
    text = re.sub(r"\s+", " ", text).strip()

    return text

def enforce_port_names(cleaned_df, port_col="INDIAN PORT", ref_df=ref_df, ref_name_col="Name", score_threshold=85):
    choices = ref_df[ref_name_col].dropna().unique().tolist()
    choices_lower = [c.lower() for c in choices]

    stats = {"exact": 0, "fuzzy": 0, "unmatched": 0}

    def best_match(x):
        if pd.isna(x):
            stats["unmatched"] += 1
            return "Unmatched"

        # Pre-clean expression
        x_norm = clean_expression(x)

        if not x_norm:
            stats["unmatched"] += 1
            return "Unmatched"

        # Step 1: Exact match
        if x_norm in ref_df["name_normalized"].values:
            stats["exact"] += 1
            return ref_df.loc[ref_df["name_normalized"] == x_norm, ref_name_col].values[0]

        # Step 2: Fuzzy match
        match = process.extractOne(x_norm, choices_lower)
        if match:
            match_str, score, _ = match
            if score >= score_threshold:
                idx = choices_lower.index(match_str)
                stats["fuzzy"] += 1
                return choices[idx]

        # Step 3: If nothing works → Unmatched
        stats["unmatched"] += 1
        return "Unmatched"

    # Apply matching
    cleaned_df[f"{port_col} (cleaned)"] = cleaned_df[port_col].apply(best_match)

    # Print stats
    total = sum(stats.values())
    print("\n Cleaning complete! Every port name standardised to reference list.")
    print(f"\n Match statistics (threshold = {score_threshold}):")
    print(f"  Exact matches   : {stats['exact']} ({stats['exact']/total:.1%})")
    print(f"  Fuzzy matches   : {stats['fuzzy']} ({stats['fuzzy']/total:.1%})")
    print(f"  Unmatched       : {stats['unmatched']} ({stats['unmatched']/total:.1%})")

    # --- Show unmatched values ---
    unmatched = cleaned_df[cleaned_df[f"{port_col} (cleaned)"] == "Unmatched"]
    if not unmatched.empty:
        print("\n Some rows could not be matched to reference port names. Please review:")
        print(unmatched[[port_col]].drop_duplicates())

    return cleaned_df

# --- Run cleaning ---
cleaned_df = enforce_port_names(cleaned_df, "INDIAN PORT", ref_df)


## foreign port

In [ ]:
# Manual replacements dictionary
manual_replacements = {
    ".NEPAL": "UNKNOWN",
    ".INDIA": "UNKNOWN",
    ".BANGALADESH": "UNKNOWN",
    "?lesund": "Ålesund",
    "?st? nad Orlic?": "Ústí nad Orlicí",
    "¿st¿ nad Orlic¿": "Ústí nad Orlicí",
    "nana": "UNKNOWN",
    "NA": "UNKNOWN"   # fixed typo (UNKOWN → UNKNOWN)
}

# Function to clean names
def clean_name(name):
    if pd.isna(name):
        return name
    name = str(name).strip()
    return manual_replacements.get(name, name)

# Apply cleaning to FOREIGN PORT column
data["FOREIGN PORT"] = data["FOREIGN PORT"].apply(clean_name)

# --- Optional: check what changed ---
changed_rows = data[data["FOREIGN PORT"].isin(manual_replacements.values())]
print(f"✅ Cleaning done! {len(changed_rows)} rows updated based on manual replacements.")


In [ ]:
# -------------------------------
# Step 1: Load reference data
# -------------------------------
official_df = pd.read_excel("/content/official_port_names.xlsx")   # has PORT_NAME, COUNTRY
countries_df = pd.read_csv(
    '/content/countries.csv',
    sep=";",
    quotechar='"',
    encoding="latin1"
)  # cols: full_name, code

# Map country name -> code
country_map = dict(zip(
    countries_df['full_name'].str.upper().str.strip(),
    countries_df['code'].str.upper().str.strip()
))

# -------------------------------
# Step 2: Clean raw data columns
# -------------------------------
data['FOREIGN PORT'] = data['FOREIGN PORT'].astype(str).str.strip().str.upper()
data['FOREIGN COUNTRY'] = data['FOREIGN COUNTRY'].astype(str).str.strip().str.upper()
data['country_code'] = data['FOREIGN COUNTRY'].map(country_map)

# -------------------------------
# Step 3: Country-aware fuzzy matching
# -------------------------------
def get_foreign_port_match(row, threshold=85):
    port = row['FOREIGN PORT']
    country_code = row['country_code']

    # Restrict candidates to ports of that country
    candidates = official_df.loc[official_df['COUNTRY'] == country_code, 'PORT_NAME']
    candidates = [c.upper() for c in candidates]

    if not candidates:
        return {"Foreign_Port_Cleaned": None, "Score": None, "Status": "NO_CANDIDATES",
                "Official_Country_Code": None}

    result = process.extractOne(port, candidates, scorer=fuzz.ratio)
    if result:
        match, score, _ = result
        if score >= threshold:
            return {"Foreign_Port_Cleaned": match, "Score": score, "Status": "OK",
                    "Official_Country_Code": country_code}
        else:
            return {"Foreign_Port_Cleaned": None, "Score": score, "Status": "LOW_CONFIDENCE",
                    "Official_Country_Code": country_code}
    else:
        return {"Foreign_Port_Cleaned": None, "Score": None, "Status": "NO_MATCH",
                "Official_Country_Code": country_code}

# Apply
results = data.apply(get_foreign_port_match, axis=1)
data = pd.concat([data, results.apply(pd.Series)], axis=1).reset_index(drop=True)

# -------------------------------
# Step 4: Extra conditions
# -------------------------------
data['Port_Changed'] = (
    data['Foreign_Port_Cleaned'].notna() &
    data['FOREIGN PORT'].ne(data['Foreign_Port_Cleaned'])
)

# Country mismatch flag (only for exact matches, not fuzzy)
data = data.merge(
    official_df[['PORT_NAME', 'COUNTRY']],
    left_on='FOREIGN PORT',
    right_on='PORT_NAME',
    how='left',
    suffixes=('', '_Official')
)

data['Country_Mismatch'] = (
    data['PORT_NAME'].notna() &
    data['country_code'].ne(data['COUNTRY'])
)

# Keep only OK matches
df_final = data[data['Status'] == "OK"].copy()

# -------------------------------
# Step 5: Match statistics
# -------------------------------
stats = data['Status'].value_counts(dropna=False).to_dict()
total = len(data)

print("\n✅ FOREIGN PORT cleaning complete.")
print(f"\n📊 Match statistics (threshold = 85):")
for status, count in stats.items():
    print(f"  {status:<15}: {count} ({count/total:.1%})")

# Show unmatched / problematic rows
unmatched = data[data['Status'].isin(["NO_MATCH", "LOW_CONFIDENCE", "NO_CANDIDATES"])]
if not unmatched.empty:
    print("\n⚠️ Some rows could not be matched confidently. Please review:")
    print(unmatched[['FOREIGN PORT', 'FOREIGN COUNTRY', 'Status', 'Score']].drop_duplicates())


## importer name

In [ ]:

# --- Step 1: Normalisation Function ---
def normalise_importer(name: str) -> str:
    if pd.isna(name):
        return "UNKNOWN"

    name = str(name).strip().upper()
    name = pd.Series(name).str.replace(r"[.,’’()/]", "", regex=True).iloc[0]
    name = pd.Series(name).str.replace(r"\s+", " ", regex=True).iloc[0]

    # Only symbols → TO ORDER
    if pd.Series(name).str.match(r"^[^\w]*$").iloc[0]:
        return "TO ORDER"

    # Normalise variations of TO ORDER
    name = pd.Series(name).str.replace(r"TO\s+(THE\s+)?ORDER(\s+OF.*)?", "TO ORDER", regex=True).iloc[0]

    # Common rules
    rules = {
        r"\bL\s*T\s*D\b": "LTD",
        r"\bLIMITED\b": "LTD",
        r"\bLIM\b": "LTD",
        r"\bP\s*V\s*T\b": "PVT",
        r"\bPRIVATE\b": "PVT",
        r"\bC\s*O\b": "CO",
        r"\bCOMPANY\b": "CO",
        r"\bINCORPORATED\b": "INC",
        r"\bP\s*L\s*C\b": "PLC",
        r"\bPUBLIC\s+LTD\b": "PLC",
        r"\bLLC\b": "LLC",
        r"\bL\s*L\s*C\b": "LLC",
        r"\bF\s*Z\s*C\b": "FZC"
    }
    for pattern, replacement in rules.items():
        name = re.sub(pattern, replacement, name)

    return name

# --- Step 2: Apply Normalisation ---
data["Importer_Clean"] = data["IMPORTER"].apply(normalise_importer)

# --- Step 3: Drop duplicates + sort ---
unique_importers = sorted(data["Importer_Clean"].drop_duplicates())

# --- Step 4: Fuzzy Matching Clusters ---
def fuzzy_group(names, threshold=85):
    groups = {}
    visited = set()

    for name in names:
        if name in visited:
            continue
        matches = process.extract(name, names, scorer=fuzz.token_sort_ratio, limit=None)
        cluster = [m[0] for m in matches if m[1] >= threshold]
        for c in cluster:
            visited.add(c)
            groups[c] = name   # anchor = first match
    return groups

groups = fuzzy_group(unique_importers)

# --- Step 5: Map back to dataframe ---
data["Importer_Final"] = data["Importer_Clean"].map(groups).fillna(data["Importer_Clean"])

# --- Step 6: Impact Summary ---
# Count how many rows were changed
changed_rows = (data["Importer_Clean"] != data["Importer_Final"]).sum()

print("Total rows:", len(data))
print("Rows changed (fuzzy-matched):", changed_rows)
print("Percentage changed:", round(changed_rows / len(data) * 100, 2), "%")

# Optional: show top 10 examples of changes
print("\nSample changes:")
print(data.loc[data["Importer_Clean"] != data["Importer_Final"],
               ["IMPORTER", "Importer_Clean", "Importer_Final"]].head(10))




## exporter name and iec


In [ ]:
# 1. Strip whitespace before calculating length or applying corrections
data['IEC'] = data['IEC'].astype(str).str.strip()

# 2. Re-calculate the length based on the cleaned string
data['len'] = data['IEC'].str.len()
unique_lengths = data['len'].unique()
print("Different lengths present after stripping whitespace:", unique_lengths)

# 3. Apply the corrections based on the new length

# a. Replace values of length 1 (likely missing/bad data) with 'UNKNOWN'
data.loc[data['len'] == 1, 'IEC'] = 'UNKNOWN'

# b. Add '0' before values of length 9
# We ensure we are only targeting the values that are NOT already 'UNKNOWN'
mask_to_pad = (data['len'] == 9) & (data['IEC'] != 'UNKNOWN')

data.loc[mask_to_pad, 'IEC'] = \
    '0' + data.loc[mask_to_pad, 'IEC'].astype(str)

# 4. (Optional but recommended) Re-calculate length one last time to confirm fix
data['len_final'] = data['IEC'].str.len()
print("\nFinal lengths after correction:", data['len_final'].unique())

In [ ]:
# Normalize exporter names
data['EXPORTER NAME'] = (
    data['EXPORTER NAME']
    .str.strip()
    .str.upper()
    .str.replace(r'[^A-Z0-9 ]', '', regex=True)   # remove punctuation
    .str.replace(r'\s+', ' ', regex=True)         # normalize spaces
)

In [ ]:
# --------------------
# IEC → Multiple exporter names
# --------------------
iec_name_counts = data.groupby('IEC')['EXPORTER NAME'].nunique().reset_index()
iec_name_counts.rename(columns={'EXPORTER NAME': 'unique_name_count'}, inplace=True)
iec_multiple_names = iec_name_counts[iec_name_counts['unique_name_count'] > 1]

# Optional: list all exporter names per IEC
iec_name_list = data.groupby('IEC')['EXPORTER NAME'].unique().reset_index()
iec_multiple_names = iec_multiple_names.merge(iec_name_list, on='IEC')

# --------------------
# Exporter name → Multiple IECs
# --------------------
name_iec_counts = data.groupby('EXPORTER NAME')['IEC'].nunique().reset_index()
name_iec_counts.rename(columns={'IEC': 'unique_iec_count'}, inplace=True)
name_multiple_iecs = name_iec_counts[name_iec_counts['unique_iec_count'] > 1]

# Optional: list all IECs per name
name_iec_list = data.groupby('EXPORTER NAME')['IEC'].unique().reset_index()
name_multiple_iecs = name_multiple_iecs.merge(name_iec_list, on='EXPORTER NAME')

# --------------------
# Save results
# --------------------
iec_multiple_names.to_excel(r"C:\Users\karti\Downloads\iec_with_multiple_names.xlsx", index=False)
name_multiple_iecs.to_excel(r"C:\Users\karti\Downloads\names_with_multiple_iecs.xlsx", index=False)

print(f"IC → multiple names: {len(iec_multiple_names)} cases")
print(f"Name → multiple IECs: {len(name_multiple_iecs)} cases")

In [ ]:
# Build canonical names (most frequent EXPORTER NAME per IEC)
canonical_names = (
    data.groupby(['IEC', 'EXPORTER NAME'])
        .size()
        .reset_index(name='count')
        .sort_values(['IEC', 'count'], ascending=[True, False])
        .drop_duplicates('IEC')
        .rename(columns={'EXPORTER NAME': 'canonical_name'})
)

# Create a dict mapping IEC -> canonical_name
iec_to_name = dict(zip(canonical_names["IEC"], canonical_names["canonical_name"]))

# Assign directly without merge
data["canonical_name"] = data["IEC"].map(iec_to_name)

In [ ]:
output_file = r"C:\Users\karti\Downloads\dataset_cleaned.csv"
data.to_csv(output_file, index=False, encoding="utf-8-sig")

# dropping duplicates


In [ ]:
data.info()

In [ ]:
# Drop duplicates from the full dataset
data = data.drop_duplicates()

# --- Optional: if you want to drop duplicates based on certain columns only ---
# data = data.drop_duplicates(subset=['SB DATE', 'FOB INR', 'QUANTITY', 'FOREIGN PORT'])

print(f"Final dataset shape after removing duplicates: {data.shape}")


In [ ]:
data.info()

# UID creation

In [ ]:
# Ensure Year and Month are clean
data['Year'] = data['SB DATE'].dt.year.astype(str).str[-2:]   # Extract year from SB DATE (last 2 digits)
data['Month'] = data['SB DATE'].dt.month.astype(str).str.zfill(2)  # Extract month (2 digits)

# Serial number for each (chapter, Year, Month) to avoid repeats across months
data['Serial'] = data.groupby(['chapter', 'Year', 'Month']).cumcount() + 1

# UID generation
data['uid'] = (
    data['Year'] +
    data['Month'] +
    "EX" +
    data['chapter'].astype(str).str.zfill(2) +    # Pad chapter code if needed
    "S" +
    data['Serial'].astype(str).str.zfill(9)       # Pad serial to 9 digits (consistent width)
)


In [ ]:
data.head()